In [1]:
#importing libraries 
import pandas as pd
import numpy as np

In [2]:
df=pd.read_csv("dirty_cafe_sales.csv")

In [3]:
df.head(20)

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,ERROR,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,UNKNOWN,UNKNOWN,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11
5,TXN_2602893,Smoothie,5,4.0,20.0,Credit Card,NaN,2023-03-31
6,TXN_4433211,UNKNOWN,3,3.0,9.0,ERROR,Takeaway,2023-10-06
7,TXN_6699534,Sandwich,4,4.0,16.0,Cash,UNKNOWN,2023-10-28
8,TXN_4717867,NaN,5,3.0,15.0,NaN,Takeaway,2023-07-28
9,TXN_2064365,Sandwich,5,4.0,20.0,NaN,In-store,2023-12-31


In [4]:
before_rows = df.shape[0]
before_missing = df.isnull().sum().sum()

In [5]:
df.describe(include='all')

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
count,10000,9667,9862,9821,9827,7421,6735,9841
unique,10000,10,7,8,19,5,4,367
top,TXN_1961373,Juice,5,3.0,6.0,Digital Wallet,Takeaway,UNKNOWN
freq,1,1171,2013,2429,979,2291,3022,159


In [6]:
df.dtypes

Transaction ID      str
Item                str
Quantity            str
Price Per Unit      str
Total Spent         str
Payment Method      str
Location            str
Transaction Date    str
dtype: object

## Data Quality Report

- Checked the dataset structure and data types.
- Identified missing values in multiple columns.
- Verified duplicate records.
- Examined numerical ranges to identify possible anomalies.

In [7]:
#Missing Data Handling
df.dropna(subset=['Transaction ID'])

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,ERROR,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,UNKNOWN,UNKNOWN,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11
...,...,...,...,...,...,...,...,...
9995,TXN_7672686,Coffee,2,2.0,4.0,NaN,UNKNOWN,2023-08-30
9996,TXN_9659401,NaN,3,NaN,3.0,Digital Wallet,NaN,2023-06-02
9997,TXN_5255387,Coffee,4,2.0,8.0,Digital Wallet,NaN,2023-03-02
9998,TXN_7695629,Cookie,3,NaN,3.0,Digital Wallet,NaN,2023-12-02


In [8]:
df.replace(['UNKNOWN','ERROR',""],np.nan,inplace=True)

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,NaN,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,NaN,NaN,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11
...,...,...,...,...,...,...,...,...
9995,TXN_7672686,Coffee,2,2.0,4.0,NaN,NaN,2023-08-30
9996,TXN_9659401,NaN,3,NaN,3.0,Digital Wallet,NaN,2023-06-02
9997,TXN_5255387,Coffee,4,2.0,8.0,Digital Wallet,NaN,2023-03-02
9998,TXN_7695629,Cookie,3,NaN,3.0,Digital Wallet,NaN,2023-12-02


In [9]:
cols=["Item","Payment Method","Location"]
df[cols] = df[cols].fillna(df[cols].mode().iloc[0])

In [10]:
cols=["Price Per Unit","Total Spent"]
df[cols] = df[cols].apply(pd.to_numeric, errors="coerce")
df[cols] = df[cols].fillna(df[cols].median())

In [11]:
df["Quantity"]=pd.to_numeric(df["Quantity"],errors="coerce")
df["Quantity"] = df["Quantity"].fillna(df["Quantity"].median())

## Missing Value Handling

- Replaced invalid entries such as "UNKNOWN" and "ERROR" with NaN.
- Missing numerical values were filled using the median.
- Missing categorical values were filled using the mode.

In [12]:
#Removing duplicates
duplicates_before=df.duplicated().sum()
print(duplicates_before)

0


## Duplicate Removal

- The dataset contains 0 duplicate rows , thus no rows were removed
- If duplicate rows were present, they could be removed using df.drop_duplicates().
  

In [13]:
df.dtypes

Transaction ID          str
Item                    str
Quantity            float64
Price Per Unit      float64
Total Spent         float64
Payment Method          str
Location                str
Transaction Date        str
dtype: object

In [14]:
#Standardisation
df["Transaction Date"]=pd.to_datetime(df["Transaction Date"])

In [15]:
df["Payment Method"] = df["Payment Method"].str.title()
df["Location"] = df["Location"].str.title()
df["Item"] = df["Item"].str.title()

## Standardization

- Converted date column into datetime format.
- Standardized text values using consistent capitalization.

In [16]:
#Outliers detection
Q1 = df["Total Spent"].quantile(0.25)
Q3 = df["Total Spent"].quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 1.5*IQR
upper = Q3 + 1.5*IQR
outliers = df[(df["Total Spent"]<lower) | (df["Total Spent"]>upper)]
print(outliers)

     Transaction ID   Item  Quantity  Price Per Unit  Total Spent  \
10      TXN_2548360  Salad       5.0             5.0         25.0   
51      TXN_6342161  Salad       5.0             5.0         25.0   
52      TXN_8914892  Juice       5.0             5.0         25.0   
96      TXN_5220895  Salad       5.0             5.0         25.0   
100     TXN_9517146  Juice       5.0             5.0         25.0   
...             ...    ...       ...             ...          ...   
9791    TXN_1232346  Salad       5.0             5.0         25.0   
9805    TXN_9506076  Salad       5.0             5.0         25.0   
9879    TXN_6393305  Salad       5.0             5.0         25.0   
9908    TXN_8922585  Salad       5.0             5.0         25.0   
9971    TXN_6120851  Salad       5.0             5.0         25.0   

      Payment Method  Location Transaction Date  
10              Cash  Takeaway       2023-11-07  
51    Digital Wallet  Takeaway       2023-01-08  
52    Digital Wallet 

## Outlier Detection

- Used the IQR method to identify extreme values.
- Outliers were reviewed before deciding whether to retain or remove them.

In [17]:
after_rows = df.shape[0]
after_missing = df.isnull().sum().sum()

In [18]:
#Summary Table
summary = pd.DataFrame({
    "Before": [before_rows, before_missing],
    "After": [after_rows, after_missing]
}, index=["Rows", "Missing Values"])

summary

,Before,After
Rows,10000,10000
Missing Values,6826,460


In [19]:
#Saving cleaned dataset to new csv file
df.to_csv("cleaned_cafe_sales.csv", index=False)